In [0]:
%pip install xgboost scikit-learn matplotlib seaborn

In [0]:
import mlflow
import mlflow.xgboost
import mlflow.sklearn
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)
from delta.tables import DeltaTable
from datetime import date
from mlflow.models.signature import infer_signature

In [0]:
dbutils.widgets.text("job_parameters", "{}")

parameters = json.loads(dbutils.widgets.get("job_parameters"))

# ── Debug ─────────────────────────────────────────────────────────────────
print(f"Raw widget value : {repr(dbutils.widgets.get('job_parameters'))}")
print(f"Parsed keys      : {list(parameters.keys())}")

# ── Identity & routing ────────────────────────────────────────────────────
catalog          = parameters.get("catalog")
_source_view     = parameters.get("source_view")
_target_table    = parameters.get("target_table")
primary_keys     = parameters.get("primary_keys")
job_id           = parameters.get("job_id")
parent_job       = parameters.get("parent_job_id", job_id)
partition        = parameters.get("partition")
default_value_flag = parameters.get("default_value_flag", True)

# ── ML params ─────────────────────────────────────────────────────────────
feature_cols      = parameters.get("feature_cols")
target_col        = parameters.get("target_col")
model_type        = parameters.get("model_type", "clv_regression")
experiment_path   = parameters.get("experiment_path")
test_size         = float(parameters.get("test_size", 0.2))
random_state      = int(parameters.get("random_state", 42))
categorical_cols  = parameters.get("categorical_cols", [])
fill_with_zero    = parameters.get("fill_with_zero", [])

# ── NEW: Log transform & cap params ───────────────────────────────────────
log_transform_target = parameters.get("log_transform_target", True)
clv_cap_value        = float(parameters.get("clv_cap_value", 500.0))

# ── Model hyperparams ─────────────────────────────────────────────────────
rf_params   = parameters.get("rf_params", {
    "n_estimators"    : 300,
    "max_depth"       : 10,
    "min_samples_leaf": 5,
    "random_state"    : random_state,
    "n_jobs"          : -1,
})

xgb_params  = parameters.get("xgb_params", {
    "n_estimators"     : 500,
    "max_depth"        : 6,
    "learning_rate"    : 0.05,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "min_child_weight" : 5,
    "gamma"            : 0.1,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 1.5,
    "eval_metric"      : "rmse",
    "random_state"     : random_state,
})

# ── Output config ─────────────────────────────────────────────────────────
score_col = parameters.get("score_col", "predicted_clv_90d")
tier_col  = parameters.get("tier_col",  "clv_tier")

# ── Validate required params ──────────────────────────────────────────────
required = {
    "catalog"        : catalog,
    "source_view"    : _source_view,
    "target_table"   : _target_table,
    "feature_cols"   : feature_cols,
    "target_col"     : target_col,
    "experiment_path": experiment_path,
    "primary_keys"   : primary_keys,
    "job_id"         : job_id,
    "partition"      : partition,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"❌ Missing required parameters: {missing}")

# ── Construct table names AFTER validation ────────────────────────────────
source_view    = f"{catalog}.{_source_view}"
target_table   = f"{catalog}.{_target_table}"
registry_table = f"{catalog}.gold.model_registry"

print("=" * 60)
print("  CLV REGRESSION — PARAMETERS LOADED")
print("=" * 60)
print(f"  Source view          : {source_view}")
print(f"  Target table         : {target_table}")
print(f"  Target col           : {target_col}")
print(f"  Feature count        : {len(feature_cols)}")
print(f"  Experiment           : {experiment_path}")
print(f"  Log transform target : {log_transform_target}")
print(f"  CLV cap value        : {clv_cap_value} BRL")
print(f"  Job ID               : {job_id}")
print(f"  Partition            : {partition}")
print("=" * 60)

In [0]:
experiment = mlflow.get_experiment_by_name(experiment_path)

if experiment is None:
    mlflow.create_experiment(experiment_path)
    print(f"✅ Created new experiment : {experiment_path}")
else:
    print(f"✅ Reusing experiment     : {experiment_path}")
    print(f"   Experiment ID         : {experiment.experiment_id}")

mlflow.set_experiment(experiment_path)
print(f"✅ MLflow tracking active")

In [0]:
df = spark.table(source_view).toPandas()

print(f"✅ Loaded {source_view}")
print(f"   Rows     : {df.shape[0]:,}")
print(f"   Columns  : {df.shape[1]}")

# ── Validate all expected columns exist ───────────────────────────────────
all_expected = feature_cols + [target_col] + primary_keys
missing_cols = [c for c in all_expected if c not in df.columns]
if missing_cols:
    raise ValueError(f"❌ Missing columns in source view: {missing_cols}")

print(f"\n✅ All expected columns present")

# ── Target distribution ───────────────────────────────────────────────────
target_vals = df[target_col].dropna()
print(f"\nTarget column  : {target_col}")
print(f"Non-null rows  : {len(target_vals):,}")
print(f"Null rows      : {df[target_col].isna().sum():,}")
print(f"\nTarget distribution (BRL):")
print(f"  Min    : {target_vals.min():.2f}")
print(f"  p25    : {target_vals.quantile(0.25):.2f}")
print(f"  Median : {target_vals.median():.2f}")
print(f"  Mean   : {target_vals.mean():.2f}")
print(f"  p75    : {target_vals.quantile(0.75):.2f}")
print(f"  p90    : {target_vals.quantile(0.90):.2f}")
print(f"  p99    : {target_vals.quantile(0.99):.2f}")
print(f"  Max    : {target_vals.max():.2f}")
print(f"  Std    : {target_vals.std():.2f}")

# ── Check for negative CLV values ─────────────────────────────────────────
neg_count = (target_vals < 0).sum()
if neg_count > 0:
    print(f"\n⚠️  {neg_count} negative CLV values found — will be clipped to 0")
else:
    print(f"\n✅ No negative CLV values")

In [0]:
import numpy as np

# ── Drop rows where target is null ────────────────────────────────────────
before = len(df)
df     = df[df[target_col].notna()].copy()
print(f"✅ Dropped {before - len(df):,} rows with null target")
print(f"   Remaining rows : {len(df):,}")

# ── Clip negatives to 0 ───────────────────────────────────────────────────
neg_count    = (df[target_col] < 0).sum()
df[target_col] = df[target_col].clip(lower=0)
if neg_count > 0:
    print(f"✅ Clipped {neg_count} negative CLV values to 0")

# ── FIX 1: Cap extreme outliers ───────────────────────────────────────────
pre_cap_max  = df[target_col].max()
capped_count = (df[target_col] > clv_cap_value).sum()
df[target_col] = df[target_col].clip(upper=clv_cap_value)

print(f"\n✅ CLV cap applied:")
print(f"   Cap value     : {clv_cap_value} BRL")
print(f"   Pre-cap max   : {pre_cap_max:.2f} BRL")
print(f"   Rows capped   : {capped_count:,}")
print(f"   Post-cap max  : {df[target_col].max():.2f} BRL")

# ── FIX 2: Log-transform target ───────────────────────────────────────────
if log_transform_target:
    df[target_col] = np.log1p(df[target_col])
    print(f"\n✅ Applied log1p transform to target:")
    print(f"   Transformed min    : {df[target_col].min():.4f}")
    print(f"   Transformed median : {df[target_col].median():.4f}")
    print(f"   Transformed mean   : {df[target_col].mean():.4f}")
    print(f"   Transformed max    : {df[target_col].max():.4f}")
    print(f"   Transformed std    : {df[target_col].std():.4f}")
else:
    print(f"\n⚠️  Log transform skipped (log_transform_target=False)")

# ── Encode categorical columns ────────────────────────────────────────────
encoders             = {}
encoded_feature_cols = feature_cols.copy()

for col in categorical_cols:
    if col in df.columns:
        le          = LabelEncoder()
        encoded_col = f"{col}_encoded"
        df[encoded_col] = le.fit_transform(df[col].astype(str))
        encoders[col]   = le
        if col in encoded_feature_cols:
            encoded_feature_cols.remove(col)
        encoded_feature_cols.append(encoded_col)
        print(f"\n✅ Encoded {col} → {encoded_col} ({df[encoded_col].nunique()} categories)")
    else:
        print(f"⚠️  Skipped {col} — not found in dataframe")

# ── Fill nulls ────────────────────────────────────────────────────────────
fill_with_median = [c for c in encoded_feature_cols if c not in fill_with_zero]

zero_cols_present   = [c for c in fill_with_zero   if c in encoded_feature_cols]
median_cols_present = [c for c in fill_with_median if c in encoded_feature_cols]

null_counts     = df[encoded_feature_cols].isna().sum()
cols_with_nulls = null_counts[null_counts > 0]

if len(cols_with_nulls) > 0:
    print(f"\n⚠️  Null values found in {len(cols_with_nulls)} column(s):")
    print(f"  {'Column':<45} {'Nulls':>8}  {'%':>6}  {'Strategy'}")
    print("  " + "-" * 72)
    for col, cnt in cols_with_nulls.items():
        strategy = "zero  " if col in fill_with_zero else "median"
        print(f"  {col:<45} {cnt:>8}  {cnt/len(df)*100:>5.2f}%  {strategy}")
else:
    print(f"\n✅ No null values in feature columns")

df[zero_cols_present]   = df[zero_cols_present].fillna(0)
df[median_cols_present] = df[median_cols_present].fillna(
    df[median_cols_present].median(numeric_only=True)
)

remaining_nulls = df[encoded_feature_cols].isna().sum().sum()
if remaining_nulls > 0:
    raise ValueError(f"❌ {remaining_nulls} nulls remain after filling")

print(f"\n✅ Zero-filled   : {len(zero_cols_present)} columns")
print(f"✅ Median-filled : {len(median_cols_present)} columns")
print(f"✅ Zero nulls remaining")

# ── Cast features to float ────────────────────────────────────────────────
X = df[encoded_feature_cols].astype(float)
y = df[target_col].astype(float)

print(f"\n{'='*55}")
print(f"  PREPROCESSING COMPLETE")
print(f"{'='*55}")
print(f"  Feature matrix          : {X.shape}")
print(f"  Target shape            : {y.shape}")
print(f"  Target mean (log scale) : {y.mean():.4f}")
print(f"  Target mean (BRL)       : {np.expm1(y).mean():.2f}")
print(f"  Target median (BRL)     : {np.expm1(y).median():.2f}")
print(f"{'='*55}")

In [0]:
from decimal import Decimal

VALID_NUMPY_KINDS  = {"i", "u", "f", "b"}
CONVERSION_PIPELINE = ["float64", "int64", "bool"]

validation_results = []
problem_cols   = []
converted_cols = []
already_valid  = []

print("=" * 75)
print(f"  COLUMN VALIDATION — {len(encoded_feature_cols)} features")
print("=" * 75)

for col in encoded_feature_cols:
    original_dtype = str(df[col].dtype)
    null_pct       = round(df[col].isna().sum() / len(df) * 100, 2)
    status         = None
    final_dtype    = original_dtype
    note           = ""

    if df[col].dtype.kind in VALID_NUMPY_KINDS:
        status = "✅ VALID"
        already_valid.append(col)

    elif df[col].dtype == object:
        non_null   = df[col].dropna()
        is_decimal = non_null.apply(lambda x: isinstance(x, Decimal)).any()
        converted  = False
        for target_type in CONVERSION_PIPELINE:
            try:
                df[col]     = pd.to_numeric(df[col], errors="raise").astype(target_type)
                status      = "🔄 CONVERTED"
                final_dtype = target_type
                note        = f"object ({'Decimal' if is_decimal else 'str'}) → {target_type}"
                converted_cols.append(col)
                converted = True
                break
            except Exception:
                continue
        if not converted:
            status = "❌ FAILED"
            problem_cols.append(col)

    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col]     = df[col].astype(np.int64) // 10**9
        status      = "🔄 CONVERTED"
        final_dtype = "int64"
        note        = "datetime64 → unix timestamp"
        converted_cols.append(col)

    else:
        try:
            df[col]     = pd.to_numeric(df[col], errors="raise").astype("float64")
            status      = "🔄 CONVERTED"
            final_dtype = "float64"
            converted_cols.append(col)
        except Exception as e:
            status = "❌ FAILED"
            note   = str(e)
            problem_cols.append(col)

    print(f"  {status:<15} {col:<40} {original_dtype:<12} → {final_dtype:<12} nulls={null_pct}%")

print(f"\n  ✅ Valid: {len(already_valid)} | 🔄 Converted: {len(converted_cols)} | ❌ Failed: {len(problem_cols)}")

if problem_cols:
    raise TypeError(
        f"\n❌ COLUMN VALIDATION FAILED\n"
        f"   Cannot convert: {problem_cols}\n"
        f"   Fix: add CAST(col AS DOUBLE) in Gold view or remove from feature_cols"
    )

X = df[encoded_feature_cols].astype(float)
y = df[target_col].astype(float)
print(f"\n✅ X shape : {X.shape}")
print(f"✅ y dtype : {y.dtype}")

In [0]:
# ── FIX 3: Stratify by CLV quantile bins ──────────────────────────────────
# Ensures train and test have similar CLV distributions
# Prevents extreme outliers landing disproportionately in one split
y_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = test_size,
    random_state = random_state,
    stratify     = y_bins
)

# ── Verify splits are balanced ────────────────────────────────────────────
train_mean_brl = np.expm1(y_train).mean() if log_transform_target else y_train.mean()
test_mean_brl  = np.expm1(y_test).mean()  if log_transform_target else y_test.mean()
train_med_brl  = np.expm1(y_train).median() if log_transform_target else y_train.median()
test_med_brl   = np.expm1(y_test).median()  if log_transform_target else y_test.median()
mean_diff_pct  = abs(train_mean_brl - test_mean_brl) / train_mean_brl * 100

print(f"Train : {X_train.shape[0]:,} rows | CLV mean: {train_mean_brl:.2f} BRL | median: {train_med_brl:.2f} BRL")
print(f"Test  : {X_test.shape[0]:,}  rows | CLV mean: {test_mean_brl:.2f} BRL | median: {test_med_brl:.2f} BRL")
print(f"\n  Mean diff between splits : {mean_diff_pct:.1f}%")

if mean_diff_pct > 15:
    print(f"⚠️  Train/test means differ by {mean_diff_pct:.1f}% — check for remaining outliers")
else:
    print(f"✅ Train/test distributions are balanced")

In [0]:
def regression_metrics(y_true, y_pred, prefix=""):
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)

    # MAPE — avoid division by zero for zero CLV customers
    nonzero_mask = y_true > 0
    mape = np.mean(
        np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask])
               / y_true[nonzero_mask])
    ) * 100 if nonzero_mask.sum() > 0 else None

    metrics = {
        f"{prefix}rmse"    : round(float(rmse), 4),
        f"{prefix}mae"     : round(float(mae),  4),
        f"{prefix}r2"      : round(float(r2),   4),
    }
    if mape is not None:
        metrics[f"{prefix}mape"] = round(float(mape), 4)

    return metrics

def print_metrics(metrics, title):
    print(f"\n{'='*50}")
    print(f"  {title}")
    print(f"{'='*50}")
    for k, v in metrics.items():
        unit = "%" if "mape" in k else " BRL" if "rmse" in k or "mae" in k else ""
        print(f"  {k:<20} : {v:.4f}{unit}")
    print(f"{'='*50}")

In [0]:
run_name_rf = f"clv_random_forest_{job_id}"

with mlflow.start_run(run_name=run_name_rf) as run_rf:

    mlflow.set_tags({
        "job_id"              : job_id,
        "parent_job"          : parent_job,
        "partition"           : str(partition),
        "source_view"         : source_view,
        "target_col"          : target_col,
        "model_type"          : "random_forest_regressor",
        "run_type"            : "regression",
        "log_transform_target": str(log_transform_target),
        "clv_cap_value"       : str(clv_cap_value),
    })

    mlflow.log_params({**rf_params,
        "train_size"          : X_train.shape[0],
        "test_size"           : X_test.shape[0],
        "n_features"          : len(encoded_feature_cols),
        "target_col"          : target_col,
        "log_transform_target": log_transform_target,
        "clv_cap_value"       : clv_cap_value,
    })

    # ── Train ─────────────────────────────────────────────────────────────
    rf_model = RandomForestRegressor(**rf_params)
    rf_model.fit(X_train, y_train)

    # ── Predict ───────────────────────────────────────────────────────────
    y_pred_rf_raw = rf_model.predict(X_test)

    # ── FIX 2b: Inverse transform back to BRL ────────────────────────────
    if log_transform_target:
        y_pred_rf  = np.expm1(np.clip(y_pred_rf_raw, 0, None))
        y_test_brl = np.expm1(y_test.values)
    else:
        y_pred_rf  = np.clip(y_pred_rf_raw, 0, None)
        y_test_brl = y_test.values

    # ── Evaluate on original BRL scale ───────────────────────────────────
    metrics_rf = regression_metrics(y_test_brl, y_pred_rf)
    mlflow.log_metrics(metrics_rf)
    mlflow.log_param("evaluation_scale", "BRL (inverse transformed)")

    # ── Cross-validation R² ───────────────────────────────────────────────
    cv_scores = cross_val_score(
        RandomForestRegressor(**rf_params),
        X, y,
        cv      = KFold(n_splits=5, shuffle=True, random_state=random_state),
        scoring = "r2"
    )
    mlflow.log_metric("cv_r2_mean", round(cv_scores.mean(), 4))
    mlflow.log_metric("cv_r2_std",  round(cv_scores.std(), 4))

    # ── Feature importance plot ───────────────────────────────────────────
    importance_df = pd.DataFrame({
        "feature"   : encoded_feature_cols,
        "importance": rf_model.feature_importances_
    }).sort_values("importance", ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
    ax.set_title(f"Top 20 Feature Importances — Random Forest CLV")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance_rf.png")
    plt.close()

    # ── Actual vs Predicted plot (BRL scale) ─────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    ax2.scatter(y_test_brl, y_pred_rf, alpha=0.3, s=10)
    max_val = max(y_test_brl.max(), y_pred_rf.max())
    ax2.plot([0, max_val], [0, max_val], "r--", label="Perfect prediction")
    ax2.set_xlabel("Actual CLV (BRL)")
    ax2.set_ylabel("Predicted CLV (BRL)")
    ax2.set_title(f"Actual vs Predicted — RF (R²={metrics_rf['r2']:.4f})")
    ax2.legend()
    plt.tight_layout()
    mlflow.log_figure(fig2, "actual_vs_predicted_rf.png")
    plt.close()

    # ── Log model ─────────────────────────────────────────────────────────
    signature     = infer_signature(X_train, rf_model.predict(X_train))
    input_example = X_train.head(5)

    mlflow.sklearn.log_model(
        rf_model, "clv_random_forest",
        signature=signature, input_example=input_example
    )
    rf_run_id = run_rf.info.run_id

    print_metrics(metrics_rf, "RANDOM FOREST RESULTS (BRL scale)")
    print(f"  CV R² (log scale) : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [0]:
run_name_xgb = f"clv_xgboost_regressor_{job_id}"

with mlflow.start_run(run_name=run_name_xgb) as run_xgb:

    mlflow.set_tags({
        "job_id"              : job_id,
        "parent_job"          : parent_job,
        "partition"           : str(partition),
        "source_view"         : source_view,
        "target_col"          : target_col,
        "model_type"          : "xgboost_regressor",
        "run_type"            : "regression",
        "log_transform_target": str(log_transform_target),
        "clv_cap_value"       : str(clv_cap_value),
    })

    mlflow.log_params({**xgb_params,
        "train_size"          : X_train.shape[0],
        "test_size"           : X_test.shape[0],
        "n_features"          : len(encoded_feature_cols),
        "target_col"          : target_col,
        "log_transform_target": log_transform_target,
        "clv_cap_value"       : clv_cap_value,
    })

    # ── Train ─────────────────────────────────────────────────────────────
    xgb_model = xgb.XGBRegressor(**xgb_params)
    xgb_model.fit(
        X_train, y_train,
        eval_set = [(X_test, y_test)],
        verbose  = False
    )

    # ── Predict ───────────────────────────────────────────────────────────
    y_pred_xgb_raw = xgb_model.predict(X_test)

    # ── FIX 2b: Inverse transform back to BRL ────────────────────────────
    if log_transform_target:
        y_pred_xgb = np.expm1(np.clip(y_pred_xgb_raw, 0, None))
        y_test_brl = np.expm1(y_test.values)
    else:
        y_pred_xgb = np.clip(y_pred_xgb_raw, 0, None)
        y_test_brl = y_test.values

    # ── Evaluate on original BRL scale ───────────────────────────────────
    metrics_xgb = regression_metrics(y_test_brl, y_pred_xgb)
    mlflow.log_metrics(metrics_xgb)
    mlflow.log_param("evaluation_scale", "BRL (inverse transformed)")

    # ── Cross-validation R² ───────────────────────────────────────────────
    cv_scores_xgb = cross_val_score(
        xgb.XGBRegressor(**xgb_params),
        X, y,
        cv      = KFold(n_splits=5, shuffle=True, random_state=random_state),
        scoring = "r2"
    )
    mlflow.log_metric("cv_r2_mean", round(cv_scores_xgb.mean(), 4))
    mlflow.log_metric("cv_r2_std",  round(cv_scores_xgb.std(), 4))

    # ── Feature importance plot ───────────────────────────────────────────
    fig3, ax3 = plt.subplots(figsize=(10, 8))
    xgb.plot_importance(
        xgb_model, ax=ax3,
        max_num_features=20,
        title="Top 20 Feature Importances — XGBoost CLV"
    )
    plt.tight_layout()
    mlflow.log_figure(fig3, "feature_importance_xgb.png")
    plt.close()

    # ── Actual vs Predicted + Residuals (BRL scale) ───────────────────────
    residuals = y_test_brl - y_pred_xgb

    fig4, (ax4, ax5) = plt.subplots(1, 2, figsize=(14, 5))

    ax4.scatter(y_test_brl, y_pred_xgb, alpha=0.3, s=10)
    max_val = max(y_test_brl.max(), y_pred_xgb.max())
    ax4.plot([0, max_val], [0, max_val], "r--", label="Perfect prediction")
    ax4.set_xlabel("Actual CLV (BRL)")
    ax4.set_ylabel("Predicted CLV (BRL)")
    ax4.set_title(f"Actual vs Predicted — XGBoost (R²={metrics_xgb['r2']:.4f})")
    ax4.legend()

    ax5.scatter(y_pred_xgb, residuals, alpha=0.3, s=10)
    ax5.axhline(0, color="red", linestyle="--")
    ax5.set_xlabel("Predicted CLV (BRL)")
    ax5.set_ylabel("Residuals (BRL)")
    ax5.set_title("Residual Plot — XGBoost")

    fig4.suptitle(f"XGBoost CLV Regressor — {source_view}")
    plt.tight_layout()
    mlflow.log_figure(fig4, "actual_vs_predicted_xgb.png")
    plt.close()

    # ── Log model ─────────────────────────────────────────────────────────
    signature_xgb = infer_signature(X_train, xgb_model.predict(X_train))

    mlflow.xgboost.log_model(
        xgb_model, "clv_xgboost_regressor",
        signature=signature_xgb,
        input_example=X_train.head(5),
        model_format="json"
    )
    xgb_run_id = run_xgb.info.run_id

    print_metrics(metrics_xgb, "XGBOOST REGRESSOR RESULTS (BRL scale)")
    print(f"  CV R² (log scale) : {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}")

In [0]:
experiment = mlflow.get_experiment_by_name(experiment_path)

runs_df = mlflow.search_runs(
    experiment_ids = [experiment.experiment_id],
    filter_string  = f"tags.job_id = '{job_id}'",
    order_by       = ["metrics.r2 DESC"]
)

if runs_df.empty:
    raise RuntimeError(f"❌ No runs found for job_id='{job_id}'")

display_cols = [c for c in [
    "tags.mlflow.runName", "metrics.rmse",
    "metrics.mae", "metrics.r2",
    "metrics.mape", "metrics.cv_r2_mean"
] if c in runs_df.columns]

print(f"\nRun comparison for job_id='{job_id}' — ranked by R²:\n")
print(runs_df[display_cols].to_string(index=False))

# ── Best model = highest R² ───────────────────────────────────────────────
best_run      = runs_df.iloc[0]
best_run_id   = best_run["run_id"]
best_run_name = best_run["tags.mlflow.runName"]
best_r2       = best_run["metrics.r2"]
best_rmse     = best_run["metrics.rmse"]

# ── Determine artifact name based on winning model ────────────────────────
if "random_forest" in best_run_name:
    best_artifact = "clv_random_forest"
    best_model    = rf_model
    load_fn       = mlflow.sklearn.load_model
else:
    best_artifact = "clv_xgboost_regressor"
    best_model    = xgb_model
    load_fn       = mlflow.xgboost.load_model

model_uri  = f"runs:/{best_run_id}/{best_artifact}"
model_name = f"{catalog}_{target_col}_regressor"

print(f"\n✅ Best run   : {best_run_name}")
print(f"   Run ID     : {best_run_id}")
print(f"   R²         : {best_r2:.4f}")
print(f"   RMSE       : {best_rmse:.2f} BRL")

# ── Register model ────────────────────────────────────────────────────────
try:
    model_details = mlflow.register_model(model_uri, model_name)
    print(f"\n✅ Model registered  : {model_details.name}")
    print(f"   Model version     : {model_details.version}")
except Exception as e:
    print(f"\n⚠️  Model registration: {str(e)}")
    model_details = None

In [0]:
now = pd.Timestamp.now().to_pydatetime()

def safe_metric(col):
    val = best_run.get(f"metrics.{col}", None)
    return float(val) if pd.notna(val) else None

registry_schema = StructType([
    StructField("model_name",       StringType(),    nullable=False),
    StructField("model_type",       StringType(),    nullable=True),
    StructField("target_col",       StringType(),    nullable=True),
    StructField("source_view",      StringType(),    nullable=True),
    StructField("run_id",           StringType(),    nullable=False),
    StructField("model_uri",        StringType(),    nullable=True),
    StructField("mlflow_version",   StringType(),    nullable=True),
    StructField("auc_roc",          DoubleType(),    nullable=True),
    StructField("f1_score",         DoubleType(),    nullable=True),
    StructField("precision_score",  DoubleType(),    nullable=True),
    StructField("recall_score",     DoubleType(),    nullable=True),
    StructField("cv_auc_mean",      DoubleType(),    nullable=True),
    StructField("cv_auc_std",       DoubleType(),    nullable=True),
    StructField("best_threshold",   DoubleType(),    nullable=True),
    StructField("score_col",        StringType(),    nullable=True),
    StructField("label_col",        StringType(),    nullable=True),
    StructField("feature_cols",     StringType(),    nullable=True),
    StructField("primary_keys",     StringType(),    nullable=True),
    StructField("job_id",           StringType(),    nullable=True),
    StructField("parent_job_id",    StringType(),    nullable=True),
    StructField("partition",        StringType(),    nullable=True),
    StructField("status",           StringType(),    nullable=True),
    StructField("registered_at",    TimestampType(), nullable=True),
    StructField("updated_at",       TimestampType(), nullable=True),
    StructField("retired_at",       TimestampType(), nullable=True),
    StructField("registered_by",    StringType(),    nullable=True),
])

registry_row = [(
    str(model_name),
    best_run.get("tags.model_type", "regressor"),
    str(target_col),
    str(source_view),
    str(best_run_id),
    str(model_uri),
    str(model_details.version) if model_details else None,
    # CLV models don't use classification metrics — store regression metrics here
    safe_metric("r2"),         # auc_roc slot → R² for regression
    safe_metric("rmse"),       # f1_score slot → RMSE for regression
    safe_metric("mae"),        # precision slot → MAE for regression
    safe_metric("mape"),       # recall slot → MAPE for regression
    safe_metric("cv_r2_mean"), # cv_auc_mean slot → CV R²
    safe_metric("cv_r2_std"),  # cv_auc_std slot → CV R² std
    None,                      # best_threshold — not applicable for regression
    str(score_col),
    str(tier_col),
    json.dumps(encoded_feature_cols),
    json.dumps(primary_keys),
    str(job_id),
    str(parent_job),
    str(partition),
    "production",
    now, now, None,
    "sahil.prusty09@gmail.com",
)]

registry_df = spark.createDataFrame(registry_row, schema=registry_schema)

DeltaTable.forName(spark, registry_table) \
    .alias("t") \
    .merge(registry_df.alias("s"), "t.model_name = s.model_name") \
    .whenMatchedUpdate(set={
        "run_id"         : "s.run_id",
        "model_uri"      : "s.model_uri",
        "mlflow_version" : "s.mlflow_version",
        "auc_roc"        : "s.auc_roc",
        "f1_score"       : "s.f1_score",
        "precision_score": "s.precision_score",
        "recall_score"   : "s.recall_score",
        "cv_auc_mean"    : "s.cv_auc_mean",
        "cv_auc_std"     : "s.cv_auc_std",
        "feature_cols"   : "s.feature_cols",
        "status"         : "s.status",
        "updated_at"     : "s.updated_at",
        "job_id"         : "s.job_id",
        "partition"      : "s.partition",
    }) \
    .whenNotMatchedInsertAll() \
    .execute()

print(f"✅ Model registry updated : {registry_table}")
print(f"\n📋 CLV registry entry:")
display(spark.table(registry_table).filter(f"model_name = '{model_name}'"))